In [ ]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Autor   : Arquitecto de Datos / MLOps Engineer
  Versión : 4.0  (Production-Ready — Server Fixes)
  Correcciones aplicadas:
      [FIX-1]  CSV con cabeceras decorativas  → comment='#' en read_csv
      [FIX-2]  Inestabilidad en Jupyter/V100  → share=True + blocking correcto
      [FIX-3]  Filtro de pandemia 2020-2021   → exclusión cronológica explícita
      [FIX-4]  Auto-detección GPU             → torch / nvidia-smi / subprocess
      [FIX-5]  Flexibilidad total de dataset  → detección dinámica de columnas
================================================================================
  Dependencias mínimas:
      pip install gradio scikit-learn pandas numpy matplotlib
  Dependencias GPU (opcionales):
      pip install xgboost lightgbm
      pip install torch  # solo para detección CUDA
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import sys
import warnings
import logging
import traceback
import subprocess
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Estado global: modelo entrenado en la sesión actual ───────────────────────
# Se rellena al finalizar entrenar() y lo consume la pestaña de Predicción.
SESION = {
    "modelo":      None,   # objeto modelo sklearn/xgb/lgbm
    "feat_cols":   [],     # lista de features en el orden correcto
    "target":      "",     # nombre del target
    "parroquia":   "",     # nombre del archivo/parroquia activa
    "feat_stats":  {},     # {"col": {"min": x, "max": x, "mean": x}}
}

# Años de pandemia a excluir del entrenamiento
ANOS_PANDEMIA = [2020, 2021]

# Colores de las gráficas (tema oscuro)
COLOR_REAL  = "#3B82F6"
COLOR_PRED  = "#F97316"
COLOR_POS   = "#22C55E"
COLOR_NEG   = "#EF4444"
BG_PLOT     = "#0F172A"
TEXT_PLOT   = "#E2E8F0"
GRID_PLOT   = "#1E293B"


# ──────────────────────────────────────────────────────────────────────────────
# 2.  [FIX-4]  DETECCIÓN AUTOMÁTICA DE GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    """
    Intenta detectar una GPU NVIDIA por tres vías independientes:
        1. nvidia-smi (disponible en el sistema operativo del servidor)
        2. torch.cuda (si PyTorch está instalado)
        3. xgboost con tree_method='hist' + device='cuda' (prueba silenciosa)
    Devuelve (gpu_disponible: bool, mensaje: str).
    """
    # Vía 1 — nvidia-smi
    try:
        resultado = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8
        )
        if resultado.returncode == 0 and resultado.stdout.strip():
            nombre_gpu = resultado.stdout.strip().split("\n")[0]
            return True, f"GPU detectada (nvidia-smi): {nombre_gpu}"
    except Exception:
        pass

    # Vía 2 — PyTorch CUDA
    try:
        import torch
        if torch.cuda.is_available():
            nombre = torch.cuda.get_device_name(0)
            return True, f"GPU detectada (torch.cuda): {nombre}"
    except ImportError:
        pass

    # Vía 3 — cuML / cupy como indicador
    try:
        import cupy  # noqa: F401
        return True, "GPU detectada (cupy disponible)"
    except ImportError:
        pass

    return False, "No se detectó GPU — se usará CPU"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


def _xgb_tree_method() -> dict:
    """Devuelve los parámetros correctos para XGBoost según la disponibilidad de GPU."""
    if GPU_DISPONIBLE:
        return {"tree_method": "hist", "device": "cuda"}   # XGBoost ≥ 2.0
    return {"tree_method": "hist", "device": "cpu"}


def _lgbm_device() -> str:
    """Devuelve 'gpu' o 'cpu' para LightGBM."""
    return "gpu" if GPU_DISPONIBLE else "cpu"


# ──────────────────────────────────────────────────────────────────────────────
# 3.  EXPLORADOR DE ARCHIVOS
# ──────────────────────────────────────────────────────────────────────────────

def listar_csvs(directorio: str = ".") -> list[str]:
    """Escanea recursivamente y devuelve todos los .csv encontrados."""
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]


def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)


# ──────────────────────────────────────────────────────────────────────────────
# 4.  [FIX-1]  CARGA DE CSV CON CABECERAS DECORATIVAS ( líneas con # )
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    """
    [FIX-1] Carga el CSV ignorando cualquier línea que empiece con '#'.
    Esto maneja cabeceras de metadatos del tipo:
        # Estación: Carapungo
        # Filas: 160,262
        # Generado: 2024-01-15
    El parámetro comment='#' le indica a pandas que descarte esas líneas
    ANTES de intentar parsear la tabla, evitando el error de columnas.
    """
    try:
        df = pd.read_csv(
            ruta,
            comment="#",         # ← FIX-1: ignora líneas decorativas con #
            low_memory=False,
            on_bad_lines="warn", # avisa en lugar de explotar por líneas malformadas
        )
        return df
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


# ──────────────────────────────────────────────────────────────────────────────
# 5.  [FIX-5]  DETECCIÓN DINÁMICA DE COLUMNAS
# ──────────────────────────────────────────────────────────────────────────────

def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    """
    [FIX-5] Detecta la columna de tiempo por nombre o contenido.
    Prioridad: keywords en el nombre → parseo exitoso de las primeras filas.
    """
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    # Fallback: buscar columna parseable como fecha
    for c in df.columns:
        try:
            sample = df[c].dropna().astype(str).iloc[:10]
            pd.to_datetime(sample, infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


def obtener_columnas_csv(ruta: str) -> tuple[list[str], str]:
    """
    [FIX-5] Carga solo las columnas de un CSV para poblar el Dropdown de target
    sin cargar todo el dataset. Devuelve (lista_de_columnas, mensaje_de_estado).
    """
    if not ruta or ruta.startswith("(No"):
        return [], "⚠️ Selecciona un archivo válido."
    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        cols = df_head.columns.tolist()
        ts = _detectar_timestamp(df_head)
        cols_sin_ts = [c for c in cols if c != ts] if ts else cols
        msg = f"✅ {len(cols)} columnas detectadas. Timestamp: `{ts}`"
        return cols_sin_ts, msg
    except Exception as e:
        return [], f"❌ Error al leer encabezados: {e}"


# ──────────────────────────────────────────────────────────────────────────────
# 6.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def _preprocesar(
    df: pd.DataFrame,
    target_col: str,
    timestamp_col: str,
    excluir_pandemia: bool = True,
) -> tuple:
    """
    Limpieza completa:
      - Convierte timestamp a índice DatetimeIndex
      - [FIX-3] Excluye años 2020-2021 (pandemia) si se solicita
      - Fuerza a numérico todas las columnas contaminantes
      - Elimina filas sin target
      - Devuelve X, y, feature_cols, índice
    """
    # Convertir y ordenar por tiempo
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()

    # ── [FIX-3]  Exclusión de pandemia (lógica correcta) ─────────────────────
    n_antes = len(df)
    if excluir_pandemia:
        mask_pandemia = df.index.year.isin(ANOS_PANDEMIA)
        df = df[~mask_pandemia]          # ~ invierte: mantiene lo que NO es pandemia
        n_excluidos = n_antes - len(df)
        if n_excluidos > 0:
            log.info(
                f"[FIX-3] Pandemia excluida: {n_excluidos:,} registros de "
                f"{ANOS_PANDEMIA} eliminados. Quedan {len(df):,} filas."
            )

    # Forzar numérico en columnas de texto (contaminantes leídos como string)
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")

    # Eliminar filas sin target
    df = df.dropna(subset=[target_col])

    # Features = todas las numéricas excepto el target
    num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in num_cols if c != target_col]

    if not feat_cols:
        raise ValueError("No se encontraron columnas numéricas para usar como features.")

    return df[feat_cols], df[target_col], feat_cols, df.index


def _dividir_cronologico(X, y, ratio=0.80):
    """División temporal estricta sin data leakage."""
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]


# ──────────────────────────────────────────────────────────────────────────────
# 7.  MODELOS
# ──────────────────────────────────────────────────────────────────────────────

def _entrenar_hgb(X_tr, y_tr) -> HistGradientBoostingRegressor:
    """
    HistGradientBoosting: maneja NaN nativamente, no requiere GPU.
    Hiperparámetros ajustados para reducir overfitting (ΔR² > 0.15):
      · max_depth=4          ← antes 6; árboles más cortos, menos memorización
      · min_samples_leaf=25  ← mínimo de muestras por hoja; fuerza generalización
      · l2_regularization=0.3 ← penalización aumentada
    """
    modelo = HistGradientBoostingRegressor(
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=30,
        validation_fraction=0.1,
        max_depth=4,               # ← req-3: reducido de 6 → 4
        min_samples_leaf=25,       # ← req-3: nuevo; evita hojas con pocos datos
        learning_rate=0.05,
        l2_regularization=0.3,     # ← req-3: aumentado de 0.1 → 0.3
        random_state=42,
    )
    modelo.fit(X_tr, y_tr)
    return modelo


def _entrenar_xgb(X_tr, y_tr):
    """
    [FIX-4] XGBoost con detección automática de GPU.
    Si GPU_DISPONIBLE=True → device='cuda', si no → device='cpu'.
    """
    try:
        import xgboost as xgb
    except ImportError:
        raise ImportError("XGBoost no instalado. Ejecuta: pip install xgboost")

    params_hw = _xgb_tree_method()
    log.info(f"XGBoost hardware params: {params_hw}")

    modelo = xgb.XGBRegressor(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="rmse",
        early_stopping_rounds=30,
        random_state=42,
        verbosity=0,
        **params_hw,                     # ← GPU o CPU según detección
    )
    modelo.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr)], verbose=False)
    return modelo


def _entrenar_lgbm(X_tr, y_tr):
    """
    [FIX-4] LightGBM con detección automática de GPU.
    """
    try:
        import lightgbm as lgb
    except ImportError:
        raise ImportError("LightGBM no instalado. Ejecuta: pip install lightgbm")

    device = _lgbm_device()
    log.info(f"LightGBM device: {device}")

    modelo = lgb.LGBMRegressor(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        device=device,                   # ← 'gpu' o 'cpu'
        random_state=42,
        verbose=-1,
    )
    modelo.fit(
        X_tr, y_tr,
        eval_set=[(X_tr, y_tr)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)],
    )
    return modelo


# ──────────────────────────────────────────────────────────────────────────────
# 8.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _fig_prediccion(y_test: pd.Series, y_pred: np.ndarray, target_col: str, days: int,
                    parroquia: str = ""):
    df_p = pd.DataFrame({"Real": y_test.values, "Predicho": y_pred}, index=y_test.index)
    inicio = df_p.index.max() - pd.Timedelta(days=days)
    df_p   = df_p[df_p.index >= inicio]

    titulo = f"Surrogate Model — {target_col}  ·  Últimos {days} días (Test)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"   # ← req-4: nombre de parroquia

    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    ax.plot(df_p.index, df_p["Real"],     label="Real",    color=COLOR_REAL, lw=1.8, alpha=0.95)
    ax.plot(df_p.index, df_p["Predicho"], label="Predicho", color=COLOR_PRED, lw=1.5,
            linestyle="--", alpha=0.90)
    ax.fill_between(df_p.index, df_p["Real"], df_p["Predicho"], alpha=0.07, color=COLOR_PRED)

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9)
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)

    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_ylabel(target_col, color=TEXT_PLOT)
    ax.set_xlabel("Fecha", color=TEXT_PLOT)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout()
    return fig


def _fig_feature_importance(fi_df: pd.DataFrame, target_col: str, parroquia: str = ""):
    n   = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
            xerr=fi_df["Std"][::-1], color=colores[::-1],
            align="center", alpha=0.85, ecolor="#94A3B8", capsize=3, height=0.65)
    ax.axvline(0, color="#475569", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)

    titulo_fi = f"Feature Importance  ·  {target_col}  (Permutation Δ R²)"
    if parroquia:
        titulo_fi = f"[{parroquia}]  {titulo_fi}"   # ← req-4

    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo_fi, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R²)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


# ── [REQ-1]  Heatmap de correlación de Pearson ───────────────────────────────

def _fig_heatmap(df_full: pd.DataFrame, target_col: str, parroquia: str = ""):
    """
    Genera un mapa de calor de correlación de Pearson entre todas las
    variables numéricas del dataset. Resalta la columna/fila del target.
    """
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return None

    corr = num_df.corr(method="pearson")
    n    = len(corr)
    fig_h = max(7, n * 0.55)
    fig_w = max(8, n * 0.60)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True   # triángulo superior oculto

    cmap = sns.diverging_palette(230, 20, as_cmap=True)   # azul–blanco–rojo

    sns.heatmap(
        corr,
        mask=mask,
        cmap=cmap,
        vmin=-1, vmax=1, center=0,
        annot=True, fmt=".2f", annot_kws={"size": 7.5, "color": TEXT_PLOT},
        linewidths=0.4, linecolor=GRID_PLOT,
        square=True,
        ax=ax,
        cbar_kws={"shrink": 0.7},
    )

    # Resaltar la fila/columna del target
    if target_col in corr.columns:
        idx = list(corr.columns).index(target_col)
        ax.add_patch(plt.Rectangle((idx, 0),    1, n, fill=False,
                                    edgecolor="#F97316", lw=2.5, clip_on=False))
        ax.add_patch(plt.Rectangle((0,   idx),  n, 1, fill=False,
                                    edgecolor="#F97316", lw=2.5, clip_on=False))

    titulo_hm = f"Correlación de Pearson — {target_col}"
    if parroquia:
        titulo_hm = f"[{parroquia}]  {titulo_hm}"   # ← req-4

    ax.set_title(titulo_hm, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)

    plt.tight_layout()
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 9.  PIPELINE PRINCIPAL  (callback del botón Entrenar)
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_local: str,
    csv_upload,
    target_col: str,
    nombre_modelo: str,
    algoritmo: str,
    plot_days: int,
    train_ratio: float,
    excluir_pandemia: bool,
):
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")

    def _estado(): return "**Registro:**  " + "  ·  ".join(logs)
    VACIO = (None, None, None, None)   # 4 salidas: métricas, pred, fi, heatmap

    try:
        # ── 9.1  Resolver archivo ─────────────────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ni subió ningún archivo.", *VACIO, _estado()

        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        target_col    = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_model"

        # [REQ-4] Extraer nombre de parroquia del nombre de archivo
        parroquia = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 9.2  Cargar CSV  [FIX-1] ──────────────────────────────────────────
        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        # ── 9.3  Detectar timestamp  [FIX-5] ──────────────────────────────────
        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ No se encontró columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Columna de tiempo: '{ts_col}'")

        # ── 9.4  Validar target ───────────────────────────────────────────────
        if target_col not in df.columns:
            cols_disp = ", ".join(df.columns.tolist())
            err(f"Target '{target_col}' no existe.")
            return (
                f"### ❌ Target **`{target_col}`** no encontrado.\n\n"
                f"**Columnas disponibles:** `{cols_disp}`",
                *VACIO, _estado()
            )

        # ── 9.5  Preprocesar  [FIX-3] ─────────────────────────────────────────
        X, y, feat_cols, idx = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        info(f"Features ({len(feat_cols)}): {feat_cols}")

        if excluir_pandemia:
            info(f"Pandemia excluida: años {ANOS_PANDEMIA} eliminados del dataset.")

        n_nulos = int(X.isna().sum().sum())
        if n_nulos:
            warn(f"{n_nulos:,} NaN en features — HGB los maneja nativamente.")

        # ── 9.6  División cronológica ──────────────────────────────────────────
        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)
        info(
            f"Train: {len(X_tr):,} ({train_ratio*100:.0f}%)  "
            f"[{X_tr.index.min().date()} → {X_tr.index.max().date()}]"
        )
        info(
            f"Test:  {len(X_te):,} ({(1-train_ratio)*100:.0f}%)  "
            f"[{X_te.index.min().date()} → {X_te.index.max().date()}]"
        )

        # ── 9.7  Entrenamiento  [FIX-4] ───────────────────────────────────────
        info(f"GPU disponible: {GPU_DISPONIBLE}  →  {GPU_MSG}")
        info(f"Algoritmo seleccionado: {algoritmo}")

        if   "XGBoost"  in algoritmo: modelo = _entrenar_xgb(X_tr, y_tr)
        elif "LightGBM" in algoritmo: modelo = _entrenar_lgbm(X_tr, y_tr)
        else:                          modelo = _entrenar_hgb(X_tr, y_tr)

        info("Entrenamiento completado.")

        # ── 9.8  Métricas ─────────────────────────────────────────────────────
        y_pred_tr = modelo.predict(X_tr)
        y_pred_te = modelo.predict(X_te)

        def _m(yt, yp):
            return dict(
                MAE  = mean_absolute_error(yt, yp),
                RMSE = float(np.sqrt(mean_squared_error(yt, yp))),
                R2   = r2_score(yt, yp),
            )

        m_tr, m_te = _m(y_tr, y_pred_tr), _m(y_te, y_pred_te)
        gap = m_tr["R2"] - m_te["R2"]
        if gap > 0.15:
            warn(f"Posible overfitting: ΔR² = {gap:.3f}")

        hw_label       = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        metricas_md = f"""
## 📊 Métricas de Evaluación — `{target_col}` · *{parroquia}*

| Métrica | 🟦 Train | 🟧 Test |
|---------|:--------:|:-------:|
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R²** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |

{"⚠️ **Posible overfitting** — ΔR² = `" + f"{gap:.3f}`" if gap > 0.15 else "✅ Sin señales de overfitting."}

---

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Algoritmo | `{algoritmo}` |
| Hardware | {hw_label} |
| Features | `{len(feat_cols)}` columnas |
| Train / Test | `{len(X_tr):,}` / `{len(X_te):,}` registros |
| Filtro pandemia | {pandemia_label} |
"""

        # ── 9.9  Feature Importance ───────────────────────────────────────────
        info("Calculando Permutation Importance…")
        perm  = permutation_importance(modelo, X_te, y_te, n_repeats=8,
                                       random_state=42, scoring="r2")
        fi_df = (
            pd.DataFrame({
                "Feature":    feat_cols,
                "Importance": perm.importances_mean,
                "Std":        perm.importances_std,
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )
        info("Feature Importance lista.")

        # ── 9.10  Persistencia  [REQ-4: nombre de parroquia en archivos] ──────
        tag = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({"modelo": modelo, "features": feat_cols, "target": target_col,
                         "parroquia": parroquia}, fh)
        info(f"Modelo guardado: {pkl_path}")

        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        pd.DataFrame([{
            "Parroquia": parroquia, "Archivo": ruta_csv, "Target": target_col,
            **{f"Train_{k}": v for k, v in m_tr.items()},
            **{f"Test_{k}":  v for k, v in m_te.items()},
            "GPU": GPU_DISPONIBLE, "Algoritmo": algoritmo,
        }]).to_csv(OUTPUT_DIR / f"{tag}_metricas.csv", index=False)

        # ── 9.11  Guardar en sesión  [REQ-2] ──────────────────────────────────
        feat_stats = {
            col: {"min": float(X[col].min()), "max": float(X[col].max()),
                  "mean": float(X[col].mean())}
            for col in feat_cols
        }
        SESION.update({
            "modelo":     modelo,
            "feat_cols":  feat_cols,
            "target":     target_col,
            "parroquia":  parroquia,
            "feat_stats": feat_stats,
        })
        info("Modelo guardado en sesión → pestaña Predicción lista.")

        # ── 9.12  Figuras ─────────────────────────────────────────────────────
        fig_pred = _fig_prediccion(y_te, y_pred_te, target_col, plot_days, parroquia)
        fig_fi   = _fig_feature_importance(fi_df, target_col, parroquia)

        # Heatmap usa el DataFrame ya preprocesado (post-pandemia, numérico)
        df_hm = pd.concat([X, y], axis=1)
        fig_hm = _fig_heatmap(df_hm, target_col, parroquia)
        info("Heatmap de correlación generado.")

        return metricas_md, fig_pred, fig_fi, fig_hm, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (
            f"### ❌ Librería faltante\n```\n{e}\n```\n"
            f"Instala con: `pip install {pkg}`",
            None, None, None, _estado()
        )
    except Exception as e:
        err(str(e))
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```", None, None, None, _estado()


# ──────────────────────────────────────────────────────────────────────────────
# 10.  CSS
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
:root {
    --bg:      #0F172A;
    --card:    #1E293B;
    --input:   #0D1525;
    --border:  #334155;
    --accent:  #6366F1;
    --accentH: #818CF8;
    --text:    #F1F5F9;
    --muted:   #94A3B8;
    --ok:      #22C55E;
    --warn:    #F59E0B;
    --err:     #EF4444;
    --r:       10px;
}
body, .gradio-container { background:var(--bg) !important; color:var(--text) !important;
    font-family:'Inter','Segoe UI',sans-serif !important; }
.gr-group, .gr-box { background:var(--card) !important;
    border:1px solid var(--border) !important; border-radius:var(--r) !important; padding:16px !important; }
input, textarea, select { background:var(--input) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:6px !important; }
label, .gr-label { color:var(--muted) !important; font-size:.8rem !important;
    font-weight:700 !important; text-transform:uppercase; letter-spacing:.05em !important; }
button.primary { background:linear-gradient(135deg,var(--accent),#7C3AED) !important;
    color:#fff !important; border:none !important; border-radius:8px !important;
    font-weight:800 !important; font-size:1rem !important; padding:12px 28px !important;
    box-shadow:0 4px 20px rgba(99,102,241,.45); transition:all .15s; }
button.primary:hover { transform:translateY(-2px); box-shadow:0 6px 28px rgba(99,102,241,.6); }
button.secondary { background:var(--card) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:8px !important; }
.gr-markdown { color:var(--text) !important; }
.gr-markdown table { border-collapse:collapse; width:100%; }
.gr-markdown th { background:#1E293B; color:var(--accentH); padding:8px 14px; border:1px solid var(--border); }
.gr-markdown td { color:var(--text); padding:7px 14px; border:1px solid var(--border); }
.gr-markdown tr:nth-child(even) td { background:#19253a; }
.gr-file { border:2px dashed var(--accent) !important; border-radius:var(--r) !important; }
.hero { text-align:center; padding:24px 0 6px; }
.hero h1 { font-size:2rem; font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.hero p { color:var(--muted); font-size:.88rem; }
.gpu-badge { display:inline-block; padding:4px 12px; border-radius:20px; font-size:.75rem;
    font-weight:700; margin-top:4px; }
.gpu-on  { background:rgba(34,197,94,.18);  color:#4ADE80; border:1px solid #22C55E; }
.gpu-off { background:rgba(99,102,241,.15); color:#A5B4FC; border:1px solid #6366F1; }
.logs-box { background:#0B1527 !important; border:1px solid #1D3557 !important;
    border-radius:8px; padding:10px 14px; font-family:'JetBrains Mono',monospace;
    font-size:.76rem; color:#7DD3FC; max-height:130px; overflow-y:auto; line-height:1.6; }
"""


# ──────────────────────────────────────────────────────────────────────────────
# 11.  CONSTRUCCIÓN DE LA UI
# ──────────────────────────────────────────────────────────────────────────────

# ── [REQ-2]  Predicción con el modelo en memoria ─────────────────────────────

def predecir_desde_sesion(valores_json: str) -> str:
    """
    Recibe un dict JSON {feature: valor} desde la UI,
    construye un DataFrame con el orden correcto de features
    y devuelve la predicción como Markdown.
    """
    if SESION["modelo"] is None:
        return "### ⚠️ No hay modelo entrenado en la sesión.\nPrimero entrena un modelo en la pestaña principal."

    try:
        import json
        vals = json.loads(valores_json)
        feat_cols = SESION["feat_cols"]
        row = {col: float(vals.get(col, SESION["feat_stats"][col]["mean"]))
               for col in feat_cols}
        X_input = pd.DataFrame([row])
        pred    = float(SESION["modelo"].predict(X_input)[0])
        target  = SESION["target"]
        parroquia = SESION["parroquia"]

        # Clasificación orientativa de PM2.5 (μg/m³)
        if target in ("PM25", "PM2.5"):
            if pred <= 12:   calidad = "🟢 **Buena**"
            elif pred <= 35: calidad = "🟡 **Moderada**"
            elif pred <= 55: calidad = "🟠 **Insalubre para grupos sensibles**"
            elif pred <= 150: calidad = "🔴 **Insalubre**"
            else:             calidad = "🟣 **Muy insalubre / Peligrosa**"
            extra = f"\n\n**Índice de calidad del aire:** {calidad}"
        else:
            extra = ""

        return (
            f"## 🔮 Predicción — `{target}` · *{parroquia}*\n\n"
            f"| Campo | Valor |\n|-------|-------|\n"
            f"| **{target} estimado** | `{pred:.3f} µg/m³` |\n"
            f"| Parroquia | `{parroquia}` |\n"
            f"| Features usadas | `{len(feat_cols)}` |"
            f"{extra}"
        )
    except Exception as e:
        return f"### ❌ Error en predicción\n```\n{traceback.format_exc()}\n```"


def _construir_sliders_prediccion():
    """
    Genera sliders dinámicos para las features guardadas en SESION.
    Se llama solo desde dentro de construir_app(), donde Gradio está activo.
    """
    sliders = []
    for col in SESION.get("feat_cols", []):
        st = SESION["feat_stats"].get(col, {"min": 0, "max": 100, "mean": 50})
        sliders.append(gr.Slider(
            label=col,
            minimum=round(st["min"], 2),
            maximum=round(st["max"], 2),
            value=round(st["mean"], 2),
            step=round((st["max"] - st["min"]) / 200, 4) or 0.01,
        ))
    return sliders


# ──────────────────────────────────────────────────────────────────────────────
# 11.  CONSTRUCCIÓN DE LA UI
# ──────────────────────────────────────────────────────────────────────────────

MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU mode — {GPU_MSG}</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo",
            secondary_hue="sky",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model — Calidad del Aire v7",
    ) as app:

        # ── Hero ──────────────────────────────────────────────────────────────
        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>Modelo sustituto interactivo · MLOps ready · v7.0</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ══════════════════════════════════
            # PANEL IZQUIERDO — CONTROLES
            # ══════════════════════════════════
            with gr.Column(scale=1, min_width=350):

                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(
                            label="CSVs detectados en el servidor",
                            choices=listar_csvs(),
                            value=None, interactive=True, scale=5,
                        )
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")

                    csv_upload = gr.File(
                        label="O arrastra / sube un CSV externo",
                        file_types=[".csv"], type="filepath",
                    )
                    btn_detectar = gr.Button("🔍 Detectar columnas del CSV", variant="secondary", size="sm")
                    info_cols = gr.Markdown("_Pulsa 'Detectar columnas' para ver las variables disponibles._")

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")
                    with gr.Row():
                        target_input = gr.Dropdown(
                            label="Variable objetivo (Target)",
                            choices=["PM25", "PM10", "NO2", "O3", "CO", "SO2"],
                            value="PM25", allow_custom_value=True, scale=3,
                        )
                        nombre_modelo_input = gr.Textbox(
                            label="Nombre del modelo (.pkl)",
                            value="surrogate_calidad_aire", scale=3,
                        )

                    algoritmo_radio = gr.Radio(
                        label="Algoritmo de entrenamiento",
                        choices=MODELOS, value=MODELOS[0],
                    )
                    excluir_pandemia_chk = gr.Checkbox(
                        label="🚫 Excluir datos de pandemia (2020–2021)",
                        value=True,
                        info="Elimina registros de 2020 y 2021 para entrenar con condiciones normales.",
                    )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Proporción de entrenamiento",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar (Test)",
                            minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button("🚀  Iniciar Entrenamiento", variant="primary", size="lg")
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(
                    value="_Esperando ejecución…_",
                    elem_classes=["logs-box"],
                )

            # ══════════════════════════════════
            # PANEL DERECHO — RESULTADOS (5 tabs)
            # ══════════════════════════════════
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs() as tabs_resultado:

                    # Tab 1: Métricas ─────────────────────────────────────────
                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Las métricas aparecerán aquí tras entrenar.*"
                        )

                    # Tab 2: Real vs Predicho ─────────────────────────────────
                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(
                            label="Serie temporal — últimos N días del test"
                        )

                    # Tab 3: Feature Importance ───────────────────────────────
                    with gr.Tab("🔍 Feature Importance"):
                        fig_fi_output = gr.Plot(
                            label="Importancia de variables (Permutation Δ R²)"
                        )

                    # Tab 4: [REQ-1] Mapa de Correlación ─────────────────────
                    with gr.Tab("📊 Análisis de Dependencias"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre todas las variables. "
                            "La fila/columna del target aparece resaltada en naranja. "
                            "Valores cercanos a ±1 indican fuerte dependencia lineal."
                        )
                        fig_hm_output = gr.Plot(
                            label="Mapa de calor — Correlación de Pearson"
                        )

                    # Tab 5: [REQ-2] Consulta / Predicción ───────────────────
                    with gr.Tab("🔮 Consulta de Calidad del Aire"):
                        gr.Markdown(
                            "Ingresa valores ambientales para obtener una estimación "
                            "del contaminante target usando el **modelo en memoria**. "
                            "Los sliders se actualizan con los rangos del dataset entrenado."
                        )

                        # Estado del modelo en sesión
                        sesion_info = gr.Markdown(
                            "_⚠️ Entrena primero un modelo para habilitar esta sección._"
                        )

                        # Contenedor de sliders dinámicos (se regenera post-entrenamiento)
                        with gr.Group():
                            gr.Markdown("##### Variables de entrada")
                            sliders_box = gr.Column()
                            # Sliders placeholder vacíos — se repoblan en refresh_pred_ui()
                            slider_componentes = [
                                gr.Number(label=f"feature_{i}", value=0,
                                          visible=False, interactive=True)
                                for i in range(30)   # máx 30 features pre-declaradas
                            ]

                        # Campo oculto: JSON para pasar valores al backend
                        json_vals = gr.Textbox(visible=False, value="{}")

                        btn_predecir = gr.Button("🔮 Predecir", variant="primary")
                        resultado_pred = gr.Markdown(
                            value="_El resultado aparecerá aquí._"
                        )

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v7.0  ·  HGB / XGBoost / LightGBM  ·  Auto-GPU  ·  Gradio
        </div>
        """)

        # ── Funciones de actualización de la pestaña Predicción ──────────────

        def _refresh_pred_ui():
            """
            Después del entrenamiento, actualiza los Number inputs con los
            rangos y medias reales del dataset. Devuelve actualizaciones
            para los 30 slots pre-declarados.
            """
            feat_cols  = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            parroquia  = SESION.get("parroquia", "")
            target     = SESION.get("target", "")

            sesion_msg = (
                f"✅ Modelo en sesión: **{target}** · *{parroquia}* · "
                f"{len(feat_cols)} features disponibles."
                if feat_cols else
                "_⚠️ Entrena primero un modelo._"
            )

            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st  = feat_stats.get(col, {"min": 0, "max": 100, "mean": 50})
                    updates.append(gr.Number(
                        label=col,
                        value=round(st["mean"], 3),
                        visible=True,
                        interactive=True,
                    ))
                else:
                    updates.append(gr.Number(visible=False))

            return [sesion_msg] + updates

        def _construir_json(*vals):
            """Empaqueta los valores de los 30 slots en JSON para predecir."""
            feat_cols = SESION.get("feat_cols", [])
            d = {feat_cols[i]: float(vals[i])
                 for i in range(min(len(feat_cols), len(vals)))
                 if vals[i] is not None}
            import json
            return json.dumps(d)

        # ── Eventos ───────────────────────────────────────────────────────────
        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])

        def _detectar_wrapper(csv_local, csv_up):
            ruta = (csv_up if isinstance(csv_up, str) else (csv_up.name if csv_up else None)) \
                   or (csv_local if csv_local and not csv_local.startswith("(No") else None)
            if not ruta:
                return gr.Dropdown(choices=["PM25"], value="PM25"), "⚠️ Selecciona un archivo primero."
            cols, msg = obtener_columnas_csv(ruta)
            choices = cols if cols else ["PM25"]
            return gr.Dropdown(choices=choices, value=choices[0] if choices else "PM25"), msg

        btn_detectar.click(
            fn=_detectar_wrapper,
            inputs=[csv_dropdown, csv_upload],
            outputs=[target_input, info_cols],
        )

        # Entrenamiento → 5 salidas + estado + refresh sliders
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_dropdown, csv_upload, target_input, nombre_modelo_input,
                algoritmo_radio, plot_days_slider, train_ratio_slider, excluir_pandemia_chk,
            ],
            outputs=[metricas_output, fig_pred_output, fig_fi_output,
                     fig_hm_output, estado_output],
        ).then(
            fn=_refresh_pred_ui,
            inputs=[],
            outputs=[sesion_info] + slider_componentes,
        )

        # Predicción: empaquetar sliders → JSON → predecir
        btn_predecir.click(
            fn=_construir_json,
            inputs=slider_componentes,
            outputs=json_vals,
        ).then(
            fn=predecir_desde_sesion,
            inputs=[json_vals],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 12.  PUNTO DE ENTRADA — RESILIENTE EN JUPYTER / SERVIDOR REMOTO  (v6)
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    """
    [FIX-v6-3] Si el túnel público falla, detecta y muestra la IP local/pública
    del servidor para que el usuario pueda acceder directamente por red.
    Intenta tres fuentes en orden de confiabilidad:
        1. socket — IP de la interfaz de red principal (siempre disponible)
        2. hostname -I — IPs asignadas al servidor (Linux)
        3. curl ipecho.net — IP pública real (requiere internet saliente)
    """
    import socket

    # Fuente 1 — IP de la interfaz principal
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip_local = s.getsockname()[0]
        s.close()
        print(f"  🖥️  IP local   : http://{ip_local}:<PUERTO>")
    except Exception:
        ip_local = None

    # Fuente 2 — todas las IPs del servidor (Linux)
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception:
        pass

    # Fuente 3 — IP pública (puede fallar si no hay salida a internet)
    try:
        ip_pub = subprocess.check_output(
            ["curl", "-s", "--max-time", "5", "https://ipecho.net/plain"],
            text=True, timeout=7,
        ).strip()
        if ip_pub:
            print(f"  🌍  IP pública  : http://{ip_pub}:<PUERTO>")
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 60)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v6.0")
    print("═" * 60)
    print(f"  Hardware  : {GPU_MSG}")
    print(f"  GPU activa: {GPU_DISPONIBLE}")
    print("  Puerto    : automático (server_port=None)")
    print("═" * 60)
    print("\n⏳ Generando link de acceso externo... por favor espera.\n")

    try:
        # ── Intento 1: con túnel público ──────────────────────────────────────
        # server_port=None  → [FIX-v6-1] Gradio elige el primer puerto libre,
        #                      evitando el error "Address already in use" en 7860.
        # share=True        → [FIX-v6-2] Solicita túnel gradio.live.
        # max_threads=40    → [FIX-v6-2] Aumenta workers para estabilidad en V100.
        # prevent_thread_lock=False → bloquea el hilo de Jupyter indefinidamente.
        app.launch(
            server_name="0.0.0.0",
            server_port=None,             # ← [FIX-v6-1] puerto automático
            share=True,                   # ← [FIX-v6-2] túnel público
            max_threads=40,               # ← [FIX-v6-2] estabilidad V100
            debug=True,
            show_error=True,
            prevent_thread_lock=False,    # ← bloquea hilo → celda no termina
            quiet=False,
        )

    except OSError as port_err:
        # Puerto ocupado incluso con None (caso muy raro, p.ej. rango agotado)
        print(f"\n⚠️  Error de puerto: {port_err}")
        print("🔄  Reintentando en puerto 7861…\n")
        app.launch(
            server_name="0.0.0.0",
            server_port=7861,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )

    except Exception as tunnel_err:
        # ── Intento 2: sin túnel — acceso directo por IP ──────────────────────
        # [FIX-v6-3] El túnel falló (firewall, timeout de red, etc.).
        # La app sigue viva en 0.0.0.0 y se imprime la IP para acceso manual.
        print(f"\n⚠️  El túnel público falló: {tunnel_err}")
        print("🔄  Relanzando sin túnel — acceso por IP directa:\n")
        _imprimir_ip_fallback()
        print("\n   Sustituye <PUERTO> por el número que aparezca en")
        print("   'Running on local URL: http://0.0.0.0:<PUERTO>'\n")

        app.launch(
            server_name="0.0.0.0",
            server_port=None,             # sigue usando puerto automático
            share=False,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )

19:18:31 [INFO] Archivo subido: dataset_ml_carapungo.csv
19:18:32 [INFO] CSV cargado: 160,262 filas × 19 columnas
19:18:32 [INFO] Columna de tiempo: 'Timestamp'
19:18:32 [INFO] [FIX-3] Pandemia excluida: 23 registros de [2020, 2021] eliminados. Quedan 160,239 filas.
19:18:32 [INFO] Features (17): ['PM25', 'PM10', 'O3', 'CO', 'NO2', 'Temperatura', 'Humedad', 'Viento_Velocidad', 'Viento_Direccion', 'Precipitacion', 'PM25_lag_1h', 'PM25_lag_3h', 'PM25_lag_24h', 'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos']
19:18:32 [INFO] Pandemia excluida: años [2020, 2021] eliminados del dataset.
19:18:32 [WARNING] 202,995 NaN en features — HGB los maneja nativamente.
19:18:32 [INFO] Train: 145,971 (95%)  [2005-03-16 → 2025-02-20]
19:18:32 [INFO] Test:  7,683 (5%)  [2025-02-20 → 2026-02-28]
19:18:32 [INFO] GPU disponible: True  →  GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
19:18:32 [INFO] Algoritmo seleccionado: LightGBM (auto GPU/CPU)
19:18:32 [ERROR] LightGBM no instalado. Ejecuta: pi